# Ablation Study — какой компонент сколько даёт?

**Цель:** доказать, что каждый из 3 компонентов (Static / Dynamic / Content) и Attention-gate реально вносят вклад.

**5 вариантов модели:**
1. `full_model` — все 3 компонента + attention (baseline для сравнения)
2. `no_dynamic` — без GRU
3. `no_content` — без TF-IDF/MLP
4. `no_attention` — uniform веса 1/3 (не learnable gate)
5. `static_only` — только NCF (как простой baseline без GRU и content)

**Конфигурация:**
- `SUBSAMPLE_FRAC=0.1` (470K train) — для скорости. Различия между вариантами видны и на subsample.
- 20 эпох каждый, early stopping patience=4
- ~6 мин/вариант на T4 → **всего ~30-40 мин**

**Требование:** запущен `01_data_pipeline.ipynb`.

**Порядок:** Runtime → T4 GPU → Run all

In [ ]:
# ── 1. Setup + Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, torch
DRIVE_DIR     = '/content/drive/MyDrive/disser'
PROCESSED_DIR = f'{DRIVE_DIR}/data/processed'
OUTPUT_DIR    = f'{DRIVE_DIR}/outputs'
os.makedirs(f'{OUTPUT_DIR}/checkpoints', exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    torch.backends.cudnn.benchmark = True

In [ ]:
# ── 2. Hyperparameters (общие для всех вариантов — fair comparison) ──────
SUBSAMPLE_FRAC = 0.1
EMBED_DIM      = 64
MAX_SEQ_LEN    = 50
BATCH_SIZE     = 2048
LR             = 1e-3
EPOCHS         = 20      # меньше чем в main, для скорости
PATIENCE       = 4
EVAL_USERS     = 5000
EVAL_NEGATIVES = 99
USE_AMP        = True

ABLATION_VARIANTS = {
    'full_model':   (True,  True,  True),    # все компоненты
    'no_dynamic':   (False, True,  True),    # без GRU
    'no_content':   (True,  False, True),    # без TF-IDF
    'no_attention': (True,  True,  False),   # uniform 1/3
    'static_only':  (False, False, False),   # только NCF
}
print(f'Variants: {list(ABLATION_VARIANTS.keys())}')
print(f'Each: {EPOCHS} epochs, patience={PATIENCE}')

In [ ]:
# ── 3. Load data + GPU pre-compute ────────────────────────────────────────
import numpy as np, pandas as pd, scipy.sparse as sp, json, gc
from pathlib import Path

P     = Path(PROCESSED_DIR)
stats = json.load(open(P / 'dataset_stats.json'))

train_df       = pd.read_parquet(P / 'train.parquet')
val_df         = pd.read_parquet(P / 'val.parquet')
test_df        = pd.read_parquet(P / 'test.parquet')
train_temporal = pd.read_parquet(P / 'train_temporal.parquet').values.astype(np.float32)
val_temporal   = pd.read_parquet(P / 'val_temporal.parquet').values.astype(np.float32)
test_temporal  = pd.read_parquet(P / 'test_temporal.parquet').values.astype(np.float32)

tfidf_sparse  = sp.load_npz(str(P / 'item_content_sparse.npz'))
seq_data      = np.load(str(P / 'user_sequences.npz'), allow_pickle=True)
user_seqs     = dict(seq_data['sequences'].item())

N_USERS     = stats['n_users']
N_ITEMS     = stats['n_items']
CONTENT_DIM = stats['content_dim']
CONTEXT_DIM = stats['context_dim']

# Subsample
if SUBSAMPLE_FRAC < 1.0:
    n_keep = int(len(train_df) * SUBSAMPLE_FRAC)
    keep_idx = np.random.RandomState(42).choice(len(train_df), n_keep, replace=False)
    train_df = train_df.iloc[keep_idx].reset_index(drop=True)
    train_temporal = train_temporal[keep_idx]

# TF-IDF на GPU
tfidf_dense = torch.from_numpy(tfidf_sparse.toarray()).half().to(DEVICE)
pad_row = torch.zeros(1, CONTENT_DIM, dtype=torch.float16, device=DEVICE)
tfidf_gpu = torch.cat([pad_row, tfidf_dense], dim=0)
del tfidf_dense

# Sequences на GPU
seq_tensor = np.zeros((N_USERS + 1, MAX_SEQ_LEN), dtype=np.int64)
len_tensor = np.ones(N_USERS + 1, dtype=np.int64)
for uid, seq in user_seqs.items():
    L = min(len(seq), MAX_SEQ_LEN)
    if L > 0:
        seq_tensor[uid, -L:] = seq[-L:]
        len_tensor[uid] = L
seq_tensor_gpu = torch.from_numpy(seq_tensor).to(DEVICE)
len_tensor_gpu = torch.from_numpy(len_tensor).to(DEVICE)

del user_seqs, seq_data, tfidf_sparse
gc.collect()
torch.cuda.empty_cache()

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
print(f'TF-IDF on GPU: {tfidf_gpu.shape}')

In [ ]:
# ── 4. Dataset & DataLoader ───────────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader

class FastRecDataset(Dataset):
    def __init__(self, df, temporal):
        self.users    = df['user_idx'].values.astype(np.int64)
        self.items    = df['item_idx'].values.astype(np.int64)
        self.temporal = temporal
    def __len__(self): return len(self.users)
    def __getitem__(self, idx):
        neg = np.random.randint(1, N_ITEMS + 1)
        return (self.users[idx], self.items[idx], neg,
                self.temporal[idx] if idx < len(self.temporal) else np.zeros(CONTEXT_DIM, np.float32))

def collate_fn(batch):
    users = torch.tensor([b[0] for b in batch], dtype=torch.long)
    pos   = torch.tensor([b[1] for b in batch], dtype=torch.long)
    neg   = torch.tensor([b[2] for b in batch], dtype=torch.long)
    ctx   = torch.tensor(np.stack([b[3] for b in batch]), dtype=torch.float32)
    return users, pos, neg, ctx

common = dict(batch_size=BATCH_SIZE, num_workers=2, pin_memory=True,
              persistent_workers=True, collate_fn=collate_fn)
train_loader = DataLoader(FastRecDataset(train_df, train_temporal), shuffle=True,  drop_last=True, **common)
val_loader   = DataLoader(FastRecDataset(val_df,   val_temporal),   shuffle=False, **common)
test_loader  = DataLoader(FastRecDataset(test_df,  test_temporal),  shuffle=False, **common)

In [ ]:
# ── 5. Model + helpers ────────────────────────────────────────────────────
import torch.nn as nn

class StaticC(nn.Module):
    def __init__(self):
        super().__init__()
        self.user_emb = nn.Embedding(N_USERS + 1, EMBED_DIM, padding_idx=0)
        self.item_emb = nn.Embedding(N_ITEMS + 1, EMBED_DIM, padding_idx=0)
        self.mlp = nn.Sequential(nn.Linear(EMBED_DIM*2, 128), nn.ReLU(), nn.Dropout(0.1), nn.Linear(128, EMBED_DIM))
    def forward(self, u, i):
        return self.mlp(torch.cat([self.user_emb(u), self.item_emb(i)], dim=-1))

class DynamicC(nn.Module):
    def __init__(self):
        super().__init__()
        self.item_emb = nn.Embedding(N_ITEMS + 1, EMBED_DIM, padding_idx=0)
        self.gru = nn.GRU(EMBED_DIM, 128, num_layers=2, batch_first=True, dropout=0.1)
        self.proj = nn.Linear(128, EMBED_DIM)
    def forward(self, seqs, lens):
        x = self.item_emb(seqs)
        packed = nn.utils.rnn.pack_padded_sequence(x, lens.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.gru(packed)
        return self.proj(h[-1])

class ContentC(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(CONTENT_DIM, 512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, EMBED_DIM))
    def forward(self, x): return self.mlp(x)

class Gate(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(CONTEXT_DIM, 64), nn.ReLU(), nn.Linear(64, 3))
    def forward(self, ctx): return torch.softmax(self.net(ctx), dim=-1)

class HybridFast(nn.Module):
    def __init__(self, use_dyn=True, use_cnt=True, use_att=True):
        super().__init__()
        self.use_dyn, self.use_cnt, self.use_att = use_dyn, use_cnt, use_att
        self.static  = StaticC()
        self.dynamic = DynamicC()
        self.content = ContentC()
        self.gate    = Gate()
        self.head    = nn.Sequential(
            nn.Linear(EMBED_DIM, EMBED_DIM // 2), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(EMBED_DIM // 2, 1))
    def forward(self, u, items, seqs, lens, content, ctx):
        h_s = self.static(u, items)
        h_d = self.dynamic(seqs, lens) if self.use_dyn else torch.zeros_like(h_s)
        h_c = self.content(content)    if self.use_cnt else torch.zeros_like(h_s)
        stack = torch.stack([h_s, h_d, h_c], dim=1)
        w = self.gate(ctx).unsqueeze(-1) if self.use_att else torch.ones(stack.size(0), 3, 1, device=stack.device) / 3
        return self.head((stack * w).sum(dim=1)).squeeze(-1)

def lookup_seqs(uids):    return seq_tensor_gpu[uids], len_tensor_gpu[uids]
def lookup_content(iids): return tfidf_gpu[iids].float()

@torch.no_grad()
def evaluate_fast(model, loader, n_users_eval=EVAL_USERS, n_neg=EVAL_NEGATIVES, k=10):
    model.eval()
    rng = np.random.default_rng(42)
    collected = {'u': [], 'pos': [], 'ctx': []}
    n_so_far = 0
    for users, pos, _neg, ctx in loader:
        take = min(len(users), n_users_eval - n_so_far)
        collected['u'].append(users[:take])
        collected['pos'].append(pos[:take])
        collected['ctx'].append(ctx[:take])
        n_so_far += take
        if n_so_far >= n_users_eval: break
    u_all   = torch.cat(collected['u']).to(DEVICE)
    pos_all = torch.cat(collected['pos']).to(DEVICE)
    ctx_all = torch.cat(collected['ctx']).to(DEVICE)
    N = u_all.size(0)

    neg_all = torch.from_numpy(rng.integers(1, N_ITEMS + 1, size=(N, n_neg), dtype=np.int64)).to(DEVICE)
    candidates = torch.cat([pos_all.unsqueeze(1), neg_all], dim=1)
    n_cand = candidates.size(1)
    seqs, lens = lookup_seqs(u_all)

    EVAL_BATCH = 512
    all_scores = torch.zeros(N, n_cand, device=DEVICE)
    for s in range(0, N, EVAL_BATCH):
        e = min(s + EVAL_BATCH, N); b = e - s
        u_e   = u_all[s:e].unsqueeze(1).expand(-1, n_cand).reshape(-1)
        i_e   = candidates[s:e].reshape(-1)
        seq_e = seqs[s:e].unsqueeze(1).expand(-1, n_cand, -1).reshape(b*n_cand, -1)
        len_e = lens[s:e].unsqueeze(1).expand(-1, n_cand).reshape(-1)
        ctx_e = ctx_all[s:e].unsqueeze(1).expand(-1, n_cand, -1).reshape(b*n_cand, -1)
        cnt_e = lookup_content(i_e)
        all_scores[s:e] = model(u_e, i_e, seq_e, len_e, cnt_e, ctx_e).reshape(b, n_cand)

    _, ranks = all_scores.sort(dim=1, descending=True)
    pos_rank = (ranks == 0).float().argmax(dim=1)
    hits     = (pos_rank < k).float()
    return {'Recall@10': hits.mean().item(),
            'NDCG@10': (hits * (1.0 / torch.log2(pos_rank.float() + 2))).mean().item()}

In [ ]:
# ── 6. Training function ──────────────────────────────────────────────────
import time
from tqdm.notebook import tqdm

def bpr_loss(pos_s, neg_s):
    return -torch.nn.functional.logsigmoid(pos_s - neg_s).mean()

def train_one_variant(variant_name, use_dyn, use_cnt, use_att):
    """Train one ablation variant. Returns dict with val/test metrics + history."""
    print(f'\n{"="*60}\n  Training: {variant_name} (dyn={use_dyn}, cnt={use_cnt}, att={use_att})\n{"="*60}')
    
    model = HybridFast(use_dyn=use_dyn, use_cnt=use_cnt, use_att=use_att).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  Params: {n_params:,}')
    
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    
    best_ndcg, best_state, patience_cnt, history = 0.0, None, 0, []
    t_start = time.time()
    
    for epoch in range(EPOCHS):
        # Train
        model.train()
        total_loss, n_batches = 0.0, 0
        for users, pos, neg, ctx in train_loader:
            users = users.to(DEVICE, non_blocking=True)
            pos   = pos.to(DEVICE, non_blocking=True)
            neg   = neg.to(DEVICE, non_blocking=True)
            ctx   = ctx.to(DEVICE, non_blocking=True)
            seqs, lens = lookup_seqs(users)
            pos_cnt = lookup_content(pos)
            neg_cnt = lookup_content(neg)
            
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                B = users.size(0)
                u_cat   = torch.cat([users, users], 0)
                i_cat   = torch.cat([pos, neg], 0)
                seq_cat = torch.cat([seqs, seqs], 0)
                len_cat = torch.cat([lens, lens], 0)
                cnt_cat = torch.cat([pos_cnt, neg_cnt], 0)
                ctx_cat = torch.cat([ctx, ctx], 0)
                scores  = model(u_cat, i_cat, seq_cat, len_cat, cnt_cat, ctx_cat)
                loss = bpr_loss(scores[:B], scores[B:])
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            n_batches += 1
        
        avg_loss = total_loss / max(n_batches, 1)
        scheduler.step()
        val_m = evaluate_fast(model, val_loader)
        history.append({'epoch': epoch+1, 'loss': avg_loss, **val_m})
        print(f'  ep {epoch+1:2d}/{EPOCHS} | loss {avg_loss:.4f} | NDCG@10 {val_m["NDCG@10"]:.4f} | Recall@10 {val_m["Recall@10"]:.4f}')
        
        if val_m['NDCG@10'] > best_ndcg:
            best_ndcg = val_m['NDCG@10']
            best_state = {k: v.clone().cpu() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f'  Early stop @ epoch {epoch+1}')
                break
    
    elapsed = time.time() - t_start
    
    # Restore best & test eval
    if best_state is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    test_m = evaluate_fast(model, test_loader, n_users_eval=10000, n_neg=99)
    
    print(f'  ⏱ Time: {elapsed/60:.1f} min')
    print(f'  📊 Best val NDCG@10: {best_ndcg:.4f}')
    print(f'  📊 Test: NDCG@10={test_m["NDCG@10"]:.4f}, Recall@10={test_m["Recall@10"]:.4f}')
    
    return {
        'variant': variant_name,
        'flags': {'use_dynamic': use_dyn, 'use_content': use_cnt, 'use_attention': use_att},
        'n_params': n_params,
        'best_val_ndcg': best_ndcg,
        'test_metrics': test_m,
        'history': history,
        'time_minutes': elapsed / 60,
    }

In [ ]:
# ── 7. Run all 5 variants ─────────────────────────────────────────────────
ablation_results = {}
t_total = time.time()

for variant_name, (use_dyn, use_cnt, use_att) in ABLATION_VARIANTS.items():
    result = train_one_variant(variant_name, use_dyn, use_cnt, use_att)
    ablation_results[variant_name] = result
    
    # Save после каждого варианта (на случай отвала Colab)
    with open(f'{OUTPUT_DIR}/ablation_results.json', 'w') as f:
        json.dump(ablation_results, f, indent=2)

elapsed_total = (time.time() - t_total) / 60
print(f'\n{"="*60}\n✅ All variants done in {elapsed_total:.1f} min total\n{"="*60}')

In [ ]:
# ── 8. Comparison table ───────────────────────────────────────────────────
print('=' * 75)
print(f'{"Variant":<18} {"Params":>12} {"Val NDCG":>10} {"Test NDCG":>10} {"Test Recall":>12} {"Δ vs full":>10}')
print('-' * 75)

full_ndcg = ablation_results['full_model']['test_metrics']['NDCG@10']
for name, r in ablation_results.items():
    test_ndcg = r['test_metrics']['NDCG@10']
    test_rec  = r['test_metrics']['Recall@10']
    delta     = (test_ndcg - full_ndcg) / full_ndcg * 100
    delta_str = f'{delta:+.1f}%' if name != 'full_model' else 'baseline'
    print(f'{name:<18} {r["n_params"]:>12,} {r["best_val_ndcg"]:>10.4f} {test_ndcg:>10.4f} {test_rec:>12.4f} {delta_str:>10}')
print('=' * 75)

print(f'\n📁 Results saved to {OUTPUT_DIR}/ablation_results.json')

In [ ]:
# ── 9. Bar chart for dissertation ─────────────────────────────────────────
import matplotlib.pyplot as plt

names = list(ablation_results.keys())
ndcg_vals = [ablation_results[n]['test_metrics']['NDCG@10'] for n in names]
recall_vals = [ablation_results[n]['test_metrics']['Recall@10'] for n in names]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#1f77b4', '#ff7f0e', '#ff7f0e', '#ff7f0e', '#d62728']

axes[0].bar(names, ndcg_vals, color=colors)
axes[0].set_ylabel('NDCG@10')
axes[0].set_title('Ablation Study — Test NDCG@10')
axes[0].axhline(y=ndcg_vals[0], color='gray', linestyle='--', alpha=0.5, label='full_model baseline')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(ndcg_vals):
    axes[0].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=9)

axes[1].bar(names, recall_vals, color=colors)
axes[1].set_ylabel('Recall@10')
axes[1].set_title('Ablation Study — Test Recall@10')
axes[1].axhline(y=recall_vals[0], color='gray', linestyle='--', alpha=0.5, label='full_model baseline')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=20)
for i, v in enumerate(recall_vals):
    axes[1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=9)

plt.tight_layout()
FIG_PATH = f'{OUTPUT_DIR}/figures/ablation_comparison.png'
os.makedirs(os.path.dirname(FIG_PATH), exist_ok=True)
plt.savefig(FIG_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'📊 Chart saved → {FIG_PATH}')